#### 1. CONFIGURACIÓN E IMPORTACIONES

In [11]:
print("--- 1. Importando librerías ---")
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from collections import Counter

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print("Librerías importadas y configuración completa.\n")

--- 1. Importando librerías ---
Librerías importadas y configuración completa.



#### 2. CARGA DE DATOS Y PREPROCESAMIENTO

In [12]:
print("--- 2. Cargando datos y aplicando preprocesamiento ---")
# Carga y preprocesamiento idénticos
file_path = '../data/youtube_comment_dataset.csv'
df = pd.read_csv(file_path)
df.dropna(subset=['Text'], inplace=True)
lemmatizer = WordNetLemmatizer(); stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    text = str(text).lower(); text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split(); clean_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(clean_tokens)
df['Processed_Text'] = df['Text'].apply(preprocess_text)
label_cols = [col for col in df.columns if col.startswith('Is')]; [df.__setitem__(col, df[col].apply(lambda x: 1 if str(x).upper() == 'TRUE' else 0)) for col in label_cols]
df['IsHate'] = df[label_cols].any(axis=1).astype(int)

# Dividir los datos
X = df['Processed_Text']
y = df['IsHate'].values
X_train, X_test, y_train, y_test = train_test_split(X.tolist(), y, test_size=0.2, random_state=42, stratify=y)
print("Carga y preprocesamiento completados.\n")

--- 2. Cargando datos y aplicando preprocesamiento ---
Carga y preprocesamiento completados.



C:\Users\Omar\AppData\Local\Temp\ipykernel_20764\1086344579.py:12: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  label_cols = [col for col in df.columns if col.startswith('Is')]; [df.__setitem__(col, df[col].apply(lambda x: 1 if str(x).upper

#### 3. CONSTRUCCIÓN DEL VOCABULARIO Y TOKENIZACIÓN

In [13]:
print("--- 3. Preparando los datos para PyTorch ---")
# En PyTorch, a menudo construimos el vocabulario manualmente
word_counts = Counter(" ".join(X_train).split())
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
vocab_to_int = {word: i+2 for i, word in enumerate(vocab)} # +2 para PAD y OOV
vocab_to_int['<PAD>'] = 0
vocab_to_int['<OOV>'] = 1

# Tokenizar los textos
X_train_int = [[vocab_to_int.get(word, vocab_to_int['<OOV>']) for word in text.split()] for text in X_train]
X_test_int = [[vocab_to_int.get(word, vocab_to_int['<OOV>']) for word in text.split()] for text in X_test]

# Padding
def pad_features(features, seq_length):
    padded_features = np.zeros((len(features), seq_length), dtype=int)
    for i, row in enumerate(features):
        padded_features[i, -len(row):] = np.array(row)[:seq_length]
    return padded_features

seq_length = 100
X_train_pad = pad_features(X_train_int, seq_length)
X_test_pad = pad_features(X_test_int, seq_length)
print("Datos tokenizados y listos para PyTorch.\n")

--- 3. Preparando los datos para PyTorch ---
Datos tokenizados y listos para PyTorch.



#### 4. CREACIÓN DE DATALOADERS

In [14]:
print("--- 4. Creando DataLoaders de PyTorch ---")
batch_size = 32

# Convertir a tensores de PyTorch
train_data = TensorDataset(torch.from_numpy(X_train_pad), torch.from_numpy(y_train))
test_data = TensorDataset(torch.from_numpy(X_test_pad), torch.from_numpy(y_test))

# Crear los DataLoaders
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_data, shuffle=False, batch_size=batch_size)
print("DataLoaders creados.\n")

--- 4. Creando DataLoaders de PyTorch ---
DataLoaders creados.



In [15]:
train_inputs, val_inputs, train_labels, val_labels = train_test_split(X_train_pad, y_train, test_size=0.2, random_state=42)

train_data = TensorDataset(torch.from_numpy(train_inputs), torch.from_numpy(train_labels))
val_data = TensorDataset(torch.from_numpy(val_inputs), torch.from_numpy(val_labels))

batch_size = 32
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
val_loader = DataLoader(val_data, shuffle=False, batch_size=batch_size)
test_loader = DataLoader(test_data, shuffle=False, batch_size=batch_size)

#### 5. DEFINICIÓN DEL MODELO LSTM EN PYTORCH

In [16]:
print("--- 5. Definiendo un modelo LSTM más robusto ---")
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers, drop_prob=0.5):
        super(SentimentLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        # Capa de Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # Capa LSTM con dropout entre capas si n_layers > 1
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, n_layers, 
                            dropout=drop_prob, batch_first=True)
        # Capa de Dropout general
        self.dropout = nn.Dropout(0.5) # Aumentamos el dropout
        # Capa lineal y de salida
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = lstm_out[:, -1, :]
        out = self.dropout(lstm_out)
        out = self.fc(out)
        sig_out = self.sigmoid(out)
        return sig_out.squeeze()

# Instanciar el modelo con parámetros que favorecen la generalización
vocab_size = len(vocab_to_int)
embedding_dim = 64   # Reducir complejidad
hidden_dim = 128     # Reducir complejidad
n_layers = 1         # Un modelo más simple para empezar
model = SentimentLSTM(vocab_size, embedding_dim, hidden_dim, n_layers)
print("Modelo LSTM definido:\n", model)

--- 5. Definiendo un modelo LSTM más robusto ---
Modelo LSTM definido:
 SentimentLSTM(
  (embedding): Embedding(3678, 64)
  (lstm): LSTM(64, 128, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


c:\Users\Omar\Desktop\trabajo\repositorios\p-x-nlp-feel-recognize\.venv\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


#### 6. ENTRENAMIENTO DEL MODELO

In [17]:
print("\n--- 6. Entrenando el modelo con validación cruzada ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 20 # Entrenamos por más épocas, pero con early stopping
clip = 5

best_val_loss = float('inf')
patience = 3 # Número de épocas a esperar si no hay mejora
patience_counter = 0

for epoch in range(epochs):
    # Bucle de entrenamiento
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device).float()
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

    # Bucle de validación
    model.eval()
    val_losses = []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device).float()
            output = model(inputs)
            val_loss = criterion(output, labels)
            val_losses.append(val_loss.item())
    
    avg_val_loss = np.mean(val_losses)
    print(f'Epoch: {epoch+1:2}/{epochs}...',
          f'Train Loss: {loss.item():.4f}...',
          f'Val Loss: {avg_val_loss:.4f}')

    # Lógica de Early Stopping
    if avg_val_loss < best_val_loss:
        print(f'Validation loss decreased ({best_val_loss:.4f} --> {avg_val_loss:.4f}). Saving model...')
        best_val_loss = avg_val_loss
        # Guardamos el estado del "mejor" modelo hasta ahora
        torch.save(model.state_dict(), 'lstm_pytorch_best.pt')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered!")
            break


--- 6. Entrenando el modelo con validación cruzada ---
Epoch:  1/20... Train Loss: 0.6638... Val Loss: 0.6965
Validation loss decreased (inf --> 0.6965). Saving model...
Epoch:  2/20... Train Loss: 0.6839... Val Loss: 0.6931
Validation loss decreased (0.6965 --> 0.6931). Saving model...
Epoch:  3/20... Train Loss: 0.6426... Val Loss: 0.6909
Validation loss decreased (0.6931 --> 0.6909). Saving model...
Epoch:  4/20... Train Loss: 0.5955... Val Loss: 0.6965
Epoch:  5/20... Train Loss: 0.4128... Val Loss: 0.7041
Epoch:  6/20... Train Loss: 0.5981... Val Loss: 0.6938
Early stopping triggered!


#### 7. EVALUACIÓN Y COMPARACIÓN

In [18]:
print("\n--- 7. Evaluación y Comparación ---")
model.eval()
all_preds = []
with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
        output = model(inputs)
        preds = torch.round(output)
        all_preds.extend(preds.cpu().numpy())

y_pred_lstm = np.array(all_preds)

print("\nReporte de Clasificación (LSTM en Test):")
print(classification_report(y_test, y_pred_lstm, target_names=['No Odio', 'Odio']))

# Cargar el mejor modelo anterior para una comparación directa
best_previous_model = joblib.load('logreg_pipeline.joblib')

# Crear el mapeo inverso de índices a palabras
int_to_vocab = {idx: word for word, idx in vocab_to_int.items()}

# Reconstruir los textos originales desde los índices
X_test_texts = [
    " ".join(int_to_vocab.get(i, "<OOV>") for i in text if i > 1)
    for text in X_test_int
]

y_pred_previous = best_previous_model.predict(X_test_texts)
f1_previous = f1_score(y_test, y_pred_previous)
f1_lstm = f1_score(y_test, y_pred_lstm)

print("\n" + "="*30)
print("     COMPARACIÓN FINAL DE F1-SCORE")
print("="*30)
print(f"  - Modelo Anterior (Logistic Regression): {f1_previous:.4f}")
print(f"  - Modelo Nuevo (LSTM - PyTorch):         {f1_lstm:.4f}")
print("="*30)

# Guardar el modelo si es mejor
if f1_lstm > f1_previous:
    print("\n¡El modelo LSTM es el nuevo campeón!")
    torch.save(model.state_dict(), 'lstm_pytorch_model.pt')
    joblib.dump(vocab_to_int, 'vocab_pytorch.joblib')
    print("Modelo guardado como 'lstm_pytorch_model.pt' y vocabulario como 'vocab_pytorch.joblib'")


--- 7. Evaluación y Comparación ---

Reporte de Clasificación (LSTM en Test):
              precision    recall  f1-score   support

     No Odio       0.60      0.81      0.69       108
        Odio       0.62      0.37      0.46        92

    accuracy                           0.60       200
   macro avg       0.61      0.59      0.58       200
weighted avg       0.61      0.60      0.58       200


     COMPARACIÓN FINAL DE F1-SCORE
  - Modelo Anterior (Logistic Regression): 0.6816
  - Modelo Nuevo (LSTM - PyTorch):         0.4626


In [19]:
print("\n--- 7. Evaluación final con el mejor modelo LSTM guardado ---")

# Cargamos el estado del mejor modelo que encontramos durante el entrenamiento
model.load_state_dict(torch.load('lstm_pytorch_best.pt'))
model.eval()

# Evaluación en el test set
all_preds = []
with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
        output = model(inputs)
        preds = torch.round(output)
        all_preds.extend(preds.cpu().numpy())
y_pred_lstm = np.array(all_preds)

print("\nReporte de Clasificación Final (MEJOR LSTM en Test):")
print(classification_report(y_test, y_pred_lstm, target_names=['No Odio', 'Odio']))

# Cargar el baseline para comparar
best_previous_model = joblib.load('logreg_pipeline.joblib')
# Reconstruir textos de test
# ...
f1_previous = f1_score(y_test, y_pred_previous) # Asumimos que y_pred_previous ya fue calculado
f1_lstm = f1_score(y_test, y_pred_lstm)

print("\n" + "="*45)
print("     COMPARACIÓN FINAL DE F1-SCORE")
print("="*45)
print(f"  - Baseline (Logistic Regression): {f1_previous:.4f}")
print(f"  - Modelo LSTM Optimizado:         {f1_lstm:.4f}")
print("="*45)

if f1_lstm > f1_previous:
    print("\n¡El modelo LSTM ahora sí es competitivo o superior!")
    print("Guardando el vocabulario junto al modelo...")
    joblib.dump(vocab_to_int, 'vocab_pytorch.joblib')
    print("Modelo 'lstm_pytorch_best.pt' y vocabulario 'vocab_pytorch.joblib' están listos.")
else:
    print("\nLa Regresión Logística sigue siendo el modelo más robusto para este dataset.")


--- 7. Evaluación final con el mejor modelo LSTM guardado ---

Reporte de Clasificación Final (MEJOR LSTM en Test):
              precision    recall  f1-score   support

     No Odio       0.57      0.87      0.69       108
        Odio       0.59      0.22      0.32        92

    accuracy                           0.57       200
   macro avg       0.58      0.54      0.50       200
weighted avg       0.58      0.57      0.52       200


     COMPARACIÓN FINAL DE F1-SCORE
  - Baseline (Logistic Regression): 0.6816
  - Modelo LSTM Optimizado:         0.3175

La Regresión Logística sigue siendo el modelo más robusto para este dataset.
